# 用 vLLM 跑 gpt-oss

> 原文：[articles/gpt-oss/run-vllm.md](https://github.com/openai/openai-cookbook/blob/main/articles/gpt-oss/run-vllm.md)



## 這篇在做什麼
用 vLLM 在有專用 GPU 的伺服器上跑 gpt-oss-20b 或 120b，開成相容 OpenAI 的 API。然後示範用 OpenAI SDK 呼叫、做 function calling、接上 Agents SDK，還有直接用 vLLM 做 sampling。

## 學完你會
- 用 uv 裝好 vLLM，並用 `vllm serve` 啟動 gpt-oss 的 API server。
- 只改 base URL，就讓既有的 OpenAI SDK 程式碼打到自己架的 gpt-oss。
- 把 gpt-oss 接進 Agents SDK，並用 `openai-harmony` 處理直接 sampling 的 prompt 和輸出。

## 開跑前
- 需要伺服器級 GPU（20b 約 16GB VRAM，120b 建議 60GB 以上）。
- 這份是給伺服器用的，Colab 不是主要場景。要在 Colab 試的話，pip 指令前加 `!`，不過 vLLM 的特殊 wheel 對環境很挑，建議參考 run-colab 那篇用 Transformers。

In [ ]:
# 環境設定：在 Colab 左側 🔑 Secrets 新增 OPENAI_API_KEY
%pip install -q -U openai openai-agents openai_harmony vllm
import os
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
except ImportError:
    pass  # 本機執行就用自己的環境變數

# 抓原 repo 的資料檔（只下載這個資料夾）
if not os.path.exists('openai-cookbook'):
    !git clone -q --depth 1 --filter=blob:none --sparse https://github.com/openai/openai-cookbook
    !cd openai-cookbook && git sparse-checkout set articles/gpt-oss
os.chdir('openai-cookbook/articles/gpt-oss')

# 用 vLLM 跑 gpt-oss

[vLLM](https://docs.vllm.ai/en/latest/) 是開源的高吞吐量推論引擎，透過優化記憶體使用和處理速度，有效率地服務大型語言模型（LLM）。這份指南會帶你用 vLLM 在伺服器上架設 **gpt-oss-20b** 或 **gpt-oss-120b**，把 gpt-oss 當成 API 提供給你的應用程式用，甚至接上 Agents SDK。

注意這份指南是給有專用 GPU（例如 NVIDIA H100）的伺服器應用。如果是在消費級 GPU 上做本機推論，請看我們的 [Ollama](https://cookbook.openai.com/articles/gpt-oss/run-locally-ollama) 或 [LM Studio](https://cookbook.openai.com/articles/gpt-oss/run-locally-lmstudio) 指南。

## 挑模型

vLLM 支援 gpt-oss 的兩種大小：

- [**`openai/gpt-oss-20b`**](https://huggingface.co/openai/gpt-oss-20b)
  - 較小的模型
  - 只需要大約 **16GB VRAM**
- [**`openai/gpt-oss-120b`**](https://huggingface.co/openai/gpt-oss-120b)
  - 我們較大的完整尺寸模型
  - 最好有 **≥60GB VRAM**
  - 單張 H100 或多 GPU 都放得下

兩個模型開箱即是 **MXFP4 量化**。

## 快速設定

1. **安裝 vLLM**  
   vLLM 建議用 [uv](https://docs.astral.sh/uv/) 管理 Python 環境，它會依你的環境挑對實作。[詳見他們的 quickstart](https://docs.vllm.ai/en/latest/getting_started/quickstart.html#installation)。建立新的虛擬環境並安裝 vLLM：

```shell
uv venv --python 3.12 --seed
source .venv/bin/activate
uv pip install --pre vllm==0.10.1+gptoss \
    --extra-index-url https://wheels.vllm.ai/gpt-oss/ \
    --extra-index-url https://download.pytorch.org/whl/nightly/cu128 \
    --index-strategy unsafe-best-match
```

2. **啟動 server 並下載模型**  
   vLLM 的 `serve` 指令會自動從 HuggingFace 下載模型，並在 `localhost:8000` 啟動相容 OpenAI 的 server。依你要的模型大小，在伺服器的 terminal 執行下面的指令。

```shell
# For 20B
vllm serve openai/gpt-oss-20b

# For 120B
vllm serve openai/gpt-oss-120b
```

## 使用 API

vLLM 提供相容 **Chat Completions 的 API** 和相容 **Responses 的 API**，所以用 OpenAI SDK 不用改太多。Python 範例如下：

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY"
)

result = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain what MXFP4 quantization is."}
    ]
)

print(result.choices[0].message.content)

response = client.responses.create(
    model="openai/gpt-oss-120b",
    instructions="You are a helfpul assistant.",
    input="Explain what MXFP4 quantization is."
)

print(response.output_text)

如果你用過 OpenAI SDK，這一看就很熟，既有的程式碼改個 base URL 應該就能跑。

## 使用 tools（function calling）

vLLM 支援 function calling，也能讓模型具備瀏覽能力。

Function calling 在 Responses 和 Chat Completions 兩種 API 都能用。

用 Chat Completions 呼叫 function 的範例：

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather in a given city",
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string"}},
                "required": ["city"]
            },
        },
    }
]

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": "What's the weather in Berlin right now?"}],
    tools=tools
)

print(response.choices[0].message)

因為模型可以在 chain-of-thought (CoT) 過程中做 tool calling，你必須把 API 回傳的 reasoning 一起帶回下一次的 tool call 請求（也就是你提供 tool 結果的那次），一直到模型給出最終答案為止。

## 整合 Agents SDK

想把 gpt-oss 用在 OpenAI 的 **Agents SDK** 嗎？

兩種 Agents SDK 都可以覆寫 OpenAI 的 base client，指向 vLLM 上你自架的模型。Python SDK 也可以改用 [LiteLLM integration](https://openai.github.io/openai-agents-python/models/litellm/) 來 proxy 到 vLLM。

Python Agents SDK 的範例如下：

```
uv pip install openai-agents
```

In [ ]:
import asyncio
from openai import AsyncOpenAI
from agents import Agent, Runner, function_tool, OpenAIResponsesModel, set_tracing_disabled

set_tracing_disabled(True)

@function_tool
def get_weather(city: str):
    print(f"[debug] getting weather for {city}")
    return f"The weather in {city} is sunny."


async def main(model: str, api_key: str):
    agent = Agent(
        name="Assistant",
        instructions="You only respond in haikus.",
        model=OpenAIResponsesModel(
            model="openai/gpt-oss-120b",
            openai_client=AsyncOpenAI(
                base_url="http://localhost:8000/v1",
                api_key="EMPTY",
            ),
        )
        tools=[get_weather],
    )

    result = await Runner.run(agent, "What's the weather in Tokyo?")
    print(result.final_output)

if __name__ == "__main__":
    asyncio.run(main())

## 用 vLLM 直接 sampling

除了用 `vllm serve` 當 API server，也可以用 vLLM 的 Python library 直接控制推論。

如果你直接用 vLLM 做 sampling，一定要確保輸入 prompt 符合 [harmony response format](https://cookbook.openai.com/article/harmony)，否則模型沒辦法正常運作。可以用 [`openai-harmony` SDK](https://github.com/openai/harmony) 處理。

```
uv pip install openai-harmony
```

之後就能用 harmony 去編碼，並解析 vLLM 的 generate 函式產生的 token。

In [ ]:
import json
from openai_harmony import (
    HarmonyEncodingName,
    load_harmony_encoding,
    Conversation,
    Message,
    Role,
    SystemContent,
    DeveloperContent,
)

from vllm import LLM, SamplingParams

# --- 1) Render the prefill with Harmony ---
encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)

convo = Conversation.from_messages(
    [
        Message.from_role_and_content(Role.SYSTEM, SystemContent.new()),
        Message.from_role_and_content(
            Role.DEVELOPER,
            DeveloperContent.new().with_instructions("Always respond in riddles"),
        ),
        Message.from_role_and_content(Role.USER, "What is the weather like in SF?"),
    ]
)

prefill_ids = encoding.render_conversation_for_completion(convo, Role.ASSISTANT)

# Harmony stop tokens (pass to sampler so they won't be included in output)
stop_token_ids = encoding.stop_tokens_for_assistant_actions()

# --- 2) Run vLLM with prefill ---
llm = LLM(
    model="openai/gpt-oss-120b",
    trust_remote_code=True,
)

sampling = SamplingParams(
    max_tokens=128,
    temperature=1,
    stop_token_ids=stop_token_ids,
)

outputs = llm.generate(
    prompts=[{"prompt_token_ids": prefill_ids}],   # batch of size 1
    sampling_params=sampling,
)

# vLLM gives you both text and token IDs
gen = outputs[0].outputs[0]
text = gen.text
output_tokens = gen.token_ids  # <-- these are the completion token IDs (no prefill)

# --- 3) Parse the completion token IDs back into structured Harmony messages ---
entries = encoding.parse_messages_from_completion_tokens(output_tokens, Role.ASSISTANT)

# 'entries' is a sequence of structured conversation entries (assistant messages, tool calls, etc.).
for message in entries:
    print(f"{json.dumps(message.to_dict())}")

## 重點整理
- vLLM 適合伺服器級 GPU，`vllm serve` 一行就能把 gpt-oss 變成相容 OpenAI 的 API。
- 因為 API 相容，既有程式碼通常只要換 base URL 就能跑自架模型。
- gpt-oss 在 CoT 中會 tool calling，多輪呼叫時要把 reasoning 一起回傳給模型。
- 接 Agents SDK 的方式是覆寫 base client，或走 LiteLLM。
- 直接用 vLLM Python library sampling 時，prompt 一定要照 Harmony 格式，否則模型會壞掉。

## 練習
1. 啟動 `vllm serve openai/gpt-oss-20b` 後，把頁面上 OpenAI SDK 範例的 prompt 換成你自己的問題，確認能收到回應。
2. 在 function calling 範例裡再加一個 tool（例如查匯率），看模型會不會依問題選對 tool。
3. 把 Agents SDK 範例加上第二個 agent，用 handoff 把特定問題轉過去。